# bibliotecas e visão inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

In [ ]:
!pip install imbalanced-learn

In [ ]:
pd.read_csv('/content/TelecomX_tratado (1).csv')

In [ ]:
df = pd.read_csv('/content/TelecomX_tratado (1).csv')

# remover colunas irrelevantes e definir alvo

In [ ]:
# visão geral
print(df.shape)
display(df.head())
print(df.info())
print(df.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
colunas_irrelevantes = ['ID_Cliente', 'ID_Compra']  # ajuste conforme sua base
df_modelo = df.drop(columns=[col for col in colunas_irrelevantes if col in df.columns]).copy()

# garantir que a variável alvo esteja em formato binário
df_modelo['Churn '] = df_modelo['Churn'].replace({
    'Sim': 1, 'Não': 0, 'Nao': 0, 'Yes': 1, 'No': 0
})

print(df_modelo['Churn'].value_counts())
print(df_modelo['Churn'].value_counts(normalize=True))

## proporção da evasão



In [ ]:
proporcao = df_modelo['Churn'].value_counts(normalize=True) * 100
print(proporcao)

plt.figure(figsize=(5,4))
sns.countplot(data=df_modelo, x='Churn')
plt.title('Proporção da evasão')
plt.xticks([0,1], ['Não evadiu', 'Evadiu'])
plt.show()

In [ ]:
X = df_modelo.drop(columns='Churn')
y = df_modelo['Churn']

X = pd.get_dummies(X, drop_first=True)

print(X.shape)
display(X.head())

## encoding rápido

In [ ]:
X = df_modelo.drop(columns='Churn')
y = df_modelo['Churn']

X = pd.get_dummies(X, drop_first=True)

print(X.shape)
display(X.head())

## análise de correlação e seleção inicial de variáveis

In [ ]:
df_corr = pd.concat([X, y], axis=1)
corr_evasao = df_corr.corr(numeric_only=True)['Churn'].sort_values(key=abs, ascending=False)

print(corr_evasao.head(15))

In [ ]:
plt.figure(figsize=(8,10))
sns.heatmap(corr_evasao.to_frame(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlação das variáveis com a evasão')
plt.show()

In [ ]:
variaveis_selecionadas = corr_evasao[abs(corr_evasao) > 0.05].index.drop('Churn')
X = X[variaveis_selecionadas]

print(f'Número de variáveis selecionadas: {X.shape[1]}')
print(X.columns.tolist())

## análises direcionadas

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df_modelo, x='Churn', y='tempo_contrato')
plt.title('Tempo de contrato x Evasão')
plt.xticks([0,1], ['Não evadiu', 'Evadiu'])
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df_modelo, x='Churn', y='total_gasto')
plt.title('Total gasto x Evasão')
plt.xticks([0,1], ['Não evadiu', 'Evadiu'])
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df_modelo, x='tempo_contrato', y='total_gasto', hue='evasao', alpha=0.6)
plt.title('Tempo de contrato x Total gasto x Evasão')
plt.show()

### treino, teste, padronização e balanceamento

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Treino:', X_train.shape, y_train.shape)
print('Teste:', X_test.shape, y_test.shape)

# balanceamento com SMOTE apenas no treino
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('Antes do balanceamento:')
print(y_train.value_counts())

print('\nDepois do balanceamento:')
print(y_train_bal.value_counts())

### Regressão Logística

In [ ]:
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline_lr.fit(X_train_bal, y_train_bal)

y_pred_lr = pipeline_lr.predict(X_test)

In [ ]:
print('REGRESSÃO LOGÍSTICA')
print('Acurácia:', accuracy_score(y_test, y_pred_lr))
print('Precisão:', precision_score(y_test, y_pred_lr))
print('Recall:', recall_score(y_test, y_pred_lr))
print('F1-score:', f1_score(y_test, y_pred_lr))
print('\nRelatório:\n', classification_report(y_test, y_pred_lr))

cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusão - Regressão Logística')
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.show()

## Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train_bal, y_train_bal)

y_pred_rf = rf.predict(X_test)

In [ ]:
print('RANDOM FOREST')
print('Acurácia:', accuracy_score(y_test, y_pred_rf))
print('Precisão:', precision_score(y_test, y_pred_rf))
print('Recall:', recall_score(y_test, y_pred_rf))
print('F1-score:', f1_score(y_test, y_pred_rf))
print('\nRelatório:\n', classification_report(y_test, y_pred_rf))

cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens')
plt.title('Matriz de Confusão - Random Forest')
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.show()

## comparação direta dos modelos

In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['Regressão Logística', 'Random Forest'],
    'Acurácia': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ],
    'Precisão': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf)
    ],
    'F1-score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf)
    ]
})

display(resultados.sort_values(by='F1-score', ascending=False))

## verificar indícios de overfitting ou underfitting

In [ ]:
# regressão logística
acc_train_lr = accuracy_score(y_train_bal, pipeline_lr.predict(X_train_bal))
acc_test_lr = accuracy_score(y_test, y_pred_lr)

# random forest
acc_train_rf = accuracy_score(y_train_bal, rf.predict(X_train_bal))
acc_test_rf = accuracy_score(y_test, y_pred_rf)

print('Regressão Logística - treino:', acc_train_lr, '| teste:', acc_test_lr)
print('Random Forest - treino:', acc_train_rf, '| teste:', acc_test_rf)

In [ ]:
!pip install imbalanced-learn

# CONCLUSÃO

Nesta etapa do trabalho, foi realizada uma análise preditiva com o objetivo de identificar padrões relacionados à evasão de clientes e construir modelos capazes de prever esse comportamento. Inicialmente, foi feita a preparação da base de dados, com a remoção de colunas irrelevantes para a modelagem, transformação da variável alvo em formato binário e aplicação de encoding nas variáveis categóricas, permitindo seu uso nos algoritmos de machine learning.

Em seguida, foi analisada a proporção da variável de evasão, com o objetivo de verificar se havia desbalanceamento entre as classes. Essa verificação é importante porque, quando há muito mais clientes não evadidos do que evadidos, os modelos podem ter dificuldade para aprender corretamente os padrões da classe minoritária. Diante disso, foi aplicado um método de balanceamento de classes no conjunto de treino, buscando melhorar a capacidade preditiva dos modelos.

Na etapa exploratória, também foi realizada uma análise de correlação entre as variáveis numéricas e a evasão, a fim de identificar quais atributos apresentavam maior associação com o cancelamento dos clientes. Além disso, foram feitas análises direcionadas entre o tempo de contrato e a evasão, bem como entre o total gasto e a evasão, utilizando gráficos como boxplots e dispersão. De forma geral, essas visualizações permitiram observar tendências relevantes, como maior evasão entre clientes com menor tempo de relacionamento e menor volume total gasto.

Para a modelagem preditiva, o conjunto de dados foi dividido em treino e teste, permitindo avaliar o desempenho dos modelos em dados não vistos anteriormente. Foram construídos dois modelos diferentes para prever a evasão dos clientes: a Regressão Logística, que exigiu padronização dos dados, e o Random Forest, que não depende desse tipo de transformação. Essa comparação foi importante para avaliar o comportamento de abordagens distintas diante do mesmo problema.

Os modelos foram avaliados por meio das métricas de acurácia, precisão, recall, F1-score e matriz de confusão. A análise dessas métricas permitiu observar não apenas a taxa geral de acertos, mas também a capacidade dos modelos em identificar corretamente os clientes com evasão, o que é especialmente importante nesse contexto. Entre os modelos testados, [inserir nome do melhor modelo] apresentou o melhor desempenho geral, com resultados mais equilibrados entre precisão e recall, refletidos principalmente no F1-score.

Também foi feita uma análise crítica quanto à possibilidade de overfitting e underfitting. Quando o desempenho no conjunto de treino é muito superior ao desempenho no conjunto de teste, isso pode indicar overfitting, ou seja, o modelo aprendeu excessivamente os padrões específicos dos dados de treino e perdeu capacidade de generalização. Por outro lado, caso o desempenho seja baixo tanto no treino quanto no teste, isso pode indicar underfitting, sugerindo que o modelo não conseguiu capturar adequadamente as relações presentes nos dados. A comparação entre os resultados de treino e teste permitiu avaliar esse comportamento em cada modelo.

Conclui-se que a análise preditiva foi capaz de identificar fatores relevantes associados à evasão de clientes e demonstrou o potencial do uso de machine learning como apoio à tomada de decisão no contexto de e-commerce. A partir desses resultados, a empresa pode direcionar estratégias mais assertivas de retenção, especialmente para clientes com maior probabilidade de evasão, contribuindo para redução de perdas e melhoria do relacionamento com a base de clientes.